In [13]:
############################################################
# STEP3.2_03_clustering.R
#
# - 入力: STEP3.2_02_mds.R の出力 (MDS 座標 CSV)
#     sub/02_mds_STEP3.2_signlessCorr/run_<run_id>/<unit>/<dataset>/
#       MDScoords_*.csv または MDS_*.csv
#
# - 出力: クラスタリング結果 + 図 (png & pdf)
#     sub/03_clustering_STEP3.2_signlessCorr/run_<run_id>/<unit>/<dataset>/
#       labels/ClusterAssign_*.csv
#       indices/k_score_*.csv
#       plots/*.png, *.pdf
############################################################

## パッケージ読み込み -------------------------------------------------------
if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }
if (!require(ggplot2)) { install.packages("ggplot2"); library(ggplot2) }
if (!require(dplyr))   { install.packages("dplyr");   library(dplyr)   }
if (!require(stringr)) { install.packages("stringr"); library(stringr) }

set.seed(42)

############################################################
# 0. パス設定 & 基本設定
############################################################

root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
sub_base  <- file.path(root_base, "sub")

step02_base <- file.path(sub_base, "02_mds_STEP3.2_signlessCorr")
step03_base <- file.path(sub_base, "03_clustering_STEP3.2_signlessCorr")
dir.create(step03_base, showWarnings = FALSE, recursive = TRUE)

# 対象データセット
datasets_variables <- c("OH", "FP")
datasets_samples   <- c("A_OH_plus_others",
                        "B_FP_plus_others",
                        "C_OH_only",
                        "D_FP_only")

# MDS 条件 (mode × dim)
mode_dims <- c("linear_top3", "linear_cumeig",
               "nonlinear_top3", "nonlinear_cumeig")

# NbClust 指標
index_list <- c("silhouette","dunn","gap","ch","db","ptbiserial")

# 固定クラスタ数
fixed_k_list <- c(10, 25, 50)

# NbClust の探索範囲 (k)
min_nc <- 2
max_nc <- 25

############################################################
# 1. run_id の取得
############################################################

# ★ Jupyter から使うときはここを書き換える：
#    02_mds で使用した run_id を指定する
# 例: override_run_id <- "20251130_153500"
override_run_id <- "20251130_153500"

get_run_id <- function() {
  # 1) override_run_id があればそれを最優先
  if (!is.na(override_run_id) &&
      is.character(override_run_id) &&
      grepl("^\\d{8}_\\d{6}$", override_run_id)) {
    message("[INFO] Using override_run_id = ", override_run_id)
    return(override_run_id)
  }
  # 2) Rscript 実行時の引数 (Rscript STEP3.2_03_clustering.R 20251130_153500)
  args <- commandArgs(trailingOnly = TRUE)
  if (length(args) >= 1 && grepl("^\\d{8}_\\d{6}$", args[1])) {
    return(args[1])
  }
  # 3) どちらも無い場合はエラーにする
  stop("Usage: Rscript STEP3.2_03_clustering.R <run_id (YYYYMMDD_HHMMSS)>\n",
       "  - or set override_run_id <- \"YYYYMMDD_HHMMSS\" at the top of this script.")
}

run_id <- get_run_id()
cat(">>> STEP3.2_03_clustering — run_id:", run_id, "\n")

step02_run_dir <- file.path(step02_base, paste0("run_", run_id))
step03_run_dir <- file.path(step03_base, paste0("run_", run_id))
dir.create(step03_run_dir, showWarnings = FALSE, recursive = TRUE)

cat(">>> STEP3.2_03_clustering input dir:  ", normalizePath(step02_run_dir), "\n")
cat(">>> STEP3.2_03_clustering output dir: ", normalizePath(step03_run_dir), "\n")

############################################################
# 2. 図を png & pdf の両方で保存するヘルパー
############################################################

save_plot_both <- function(p, base_path, width = 6, height = 6, dpi = 300) {
  png_file <- paste0(base_path, ".png")
  pdf_file <- paste0(base_path, ".pdf")
  
  # PNG
  tryCatch({
    ggsave(filename = png_file, plot = p,
           width = width, height = height, dpi = dpi)
  }, error = function(e) {
    warning("[WARN] Failed to save PNG: ", png_file, " (", e$message, ")")
  })
  
  # PDF
  tryCatch({
    ggsave(filename = pdf_file, plot = p,
           width = width, height = height)
  }, error = function(e) {
    warning("[WARN] Failed to save PDF: ", pdf_file, " (", e$message, ")")
  })
}

############################################################
# 3. MDS 座標ファイルの読み込みヘルパー
############################################################

# cond: "linear_top3" など
read_coords_for_condition <- function(unit, dataset, cond) {
  parts    <- strsplit(cond, "_")[[1]]
  mode     <- parts[1]   # "linear" or "nonlinear"
  dim_mode <- parts[2]   # "top3" or "cumeig"
  
  mds_dir_ds <- file.path(step02_run_dir, unit, dataset)
  if (!dir.exists(mds_dir_ds)) {
    stop("[ERROR] MDS dir not found: ", mds_dir_ds)
  }
  
  # 新しい命名 (MDScoords_...) 例：
  #  variables/FP/MDScoords_linear_top3_variables_FP_20251130_153500.csv
  coord_file1 <- file.path(
    mds_dir_ds,
    sprintf("MDScoords_%s_%s_%s_%s.csv",
            cond, unit, dataset, run_id)
  )
  
  # 古い命名 (MDS_...) 例：
  #  variables/FP/MDS_linear_top3_FP_20251130_153500.csv
  #  samples/A_OH_plus_others/MDS_nonlinear_cumeig_A_OH_plus_others_20251130_153500.csv
  coord_file2 <- file.path(
    mds_dir_ds,
    sprintf("MDS_%s_%s_%s.csv",
            cond, dataset, run_id)
  )
  
  if (file.exists(coord_file1)) {
    message("  [INFO] Using coord_file (", cond, "): ", coord_file1)
    df <- read.csv(coord_file1, row.names = 1, check.names = FALSE)
  } else if (file.exists(coord_file2)) {
    message("  [INFO] Using coord_file (", cond, "): ", coord_file2)
    df <- read.csv(coord_file2, row.names = 1, check.names = FALSE)
  } else {
    warning(
      "[WARN] coord file not found for unit=", unit,
      ", ds=", dataset, ", cond=", cond, "; skip this condition.\n",
      "  tried:\n    ", coord_file1, "\n    ", coord_file2
    )
    return(NULL)
  }
  
  return(as.matrix(df))
}


############################################################
# 4. プロット作成ヘルパー
############################################################

make_cluster_plot <- function(coords, cluster_vec, title = "", subtitle = "") {
  df_plot <- data.frame(
    ID = rownames(coords),
    x  = coords[, 1],
    y  = coords[, 2],
    Cluster = factor(cluster_vec)
  )
  
  p <- ggplot(df_plot, aes(x = x, y = y, color = Cluster)) +
    geom_point(size = 1.5, alpha = 0.8) +
    # クラスタ楕円（点数が少ないと警告が出るが、そのまま許容）
    stat_ellipse(aes(group = Cluster), type = "norm", linetype = 2, alpha = 0.5) +
    theme_bw() +
    labs(title = title, subtitle = subtitle,
         x = "MDS Dim 1", y = "MDS Dim 2") +
    theme(legend.position = "right")
  
  return(p)
}

############################################################
# 5. 1データセットのクラスタリング処理
############################################################

process_one_dataset_clust <- function(unit, dataset) {
  message("\n[STEP3.2_03] unit = ", unit, ", dataset = ", dataset, "\n\n")
  
  # 出力ディレクトリ
  out_dir_ds   <- file.path(step03_run_dir, unit, dataset)
  labels_dir   <- file.path(out_dir_ds, "labels")
  indices_dir  <- file.path(out_dir_ds, "indices")
  plots_dir    <- file.path(out_dir_ds, "plots")
  dir.create(labels_dir,  showWarnings = FALSE, recursive = TRUE)
  dir.create(indices_dir, showWarnings = FALSE, recursive = TRUE)
  dir.create(plots_dir,   showWarnings = FALSE, recursive = TRUE)
  
  # 各 mode_dim ごとの処理
  for (cond in mode_dims) {
    parts    <- strsplit(cond, "_")[[1]]
    mode     <- parts[1]   # "linear" / "nonlinear"
    dim_mode <- parts[2]   # "top3" / "cumeig"
    cond_tag <- sprintf("%s_%s", mode, dim_mode)
    
    coords <- read_coords_for_condition(unit, dataset, cond)
    if (is.null(coords)) {
      next
    }
    # NbClust は 2次元を想定
    if (ncol(coords) < 2) {
      warning("[WARN] Less than 2 dimensions for ", cond_tag,
              " (unit=",unit,", ds=",dataset,"); skip.")
      next
    }
    coords_2d <- coords[, 1:2, drop = FALSE]
    
    #-------------------------------------------------------
    # 5-1. NbClust による「指標別ベスト k」探索 & 図出力
    #-------------------------------------------------------
    for (idx in index_list) {
      message("    [NbClust] index = ", idx)
      
      # NbClust 実行
      nc_res <- NULL
      tryCatch({
        nc_res <- NbClust(
          data   = coords_2d,
          distance = "euclidean",
          min.nc   = min_nc,
          max.nc   = max_nc,
          method   = "ward.D2",
          index    = idx
        )
      }, error = function(e) {
        warning("[WARN] NbClust failed for unit=",unit,
                ", ds=",dataset,", cond=",cond_tag,", index=",idx,
                " : ", e$message)
      })
      
      if (is.null(nc_res)) {
        next
      }
      
      # best partition & best k
      clusters <- nc_res$Best.partition
      best_k   <- length(unique(clusters))
      
      # ラベル保存
      df_lab <- data.frame(
        ID      = rownames(coords_2d),
        Cluster = as.integer(clusters),
        stringsAsFactors = FALSE
      )
      lab_file <- file.path(
        labels_dir,
        sprintf("ClusterAssign_%s_%s_%s_%s_%s_%s.csv",
                mode, dim_mode, idx, unit, dataset, run_id)
      )
      write.csv(df_lab, lab_file, row.names = FALSE)
      
      # k-score 保存 (NbClust の results$nc があれば)
      # index の値は case によって異なるため、簡易に扱う
      if (!is.null(nc_res$All.index)) {
        # All.index は k ごとのスコアを含む行列 (行: 指標 or k)
        # ここでは NbClust の results をそのまま使わず、
        # 「Best.nc」や「nc」の情報が無いケースもあるので簡易にスキップ可
      }
      # 以前のコード構成では、k_score_* は別ロジックだったが
      # ここでは「Best.nc を 1行表として記録」しておく
      kscore_file <- file.path(
        indices_dir,
        sprintf("k_score_%s_%s_%s_%s_%s.csv",
                cond_tag, unit, dataset, idx, run_id)
      )
      df_k <- data.frame(
        k     = best_k,
        score = NA_real_,
        stringsAsFactors = FALSE
      )
      write.csv(df_k, kscore_file, row.names = FALSE)
      
      # 図の作成 & 保存 (png & pdf)
      title_txt    <- sprintf("%s (%s, %s, %s)", dataset, unit, cond_tag, idx)
      subtitle_txt <- sprintf("NbClust best k = %d", best_k)
      p_idx <- make_cluster_plot(coords_2d, clusters,
                                 title = title_txt,
                                 subtitle = subtitle_txt)
      
      base_plot_path <- file.path(
        plots_dir,
        sprintf("ClusterPlot_%s_%s_%s_%s_%s",
                mode, dim_mode, idx, unit, dataset)
      )
      save_plot_both(p_idx, base_plot_path)
    }
    
    #-------------------------------------------------------
    # 5-2. 固定 k (10, 25, 50) のクラスタリング & 図出力
    #-------------------------------------------------------
    for (k_fixed in fixed_k_list) {
      message("    [fixed-k] k = ", k_fixed)
      
      # 距離行列 → hclust
      dist_mat <- dist(coords_2d, method = "euclidean")
      hc <- hclust(dist_mat, method = "ward.D2")
      clusters_k <- cutree(hc, k = k_fixed)
      
      df_lab_k <- data.frame(
        ID      = rownames(coords_2d),
        Cluster = as.integer(clusters_k),
        stringsAsFactors = FALSE
      )
      lab_file_k <- file.path(
        labels_dir,
        sprintf("ClusterAssign_%s_%s_k%d_%s_%s_%s.csv",
                mode, dim_mode, k_fixed, unit, dataset, run_id)
      )
      write.csv(df_lab_k, lab_file_k, row.names = FALSE)
      
      # 図の作成 & 保存 (png & pdf)
      title_txt_k    <- sprintf("%s (%s, %s, k=%d)", dataset, unit, cond_tag, k_fixed)
      subtitle_txt_k <- "Ward.D2 (fixed k)"
      p_k <- make_cluster_plot(coords_2d, clusters_k,
                               title = title_txt_k,
                               subtitle = subtitle_txt_k)
      
      base_plot_path_k <- file.path(
        plots_dir,
        sprintf("ClusterPlot_%s_%s_k%d_%s_%s",
                mode, dim_mode, k_fixed, unit, dataset)
      )
      save_plot_both(p_k, base_plot_path_k)
    }
  }
}

############################################################
# 6. メインループ
############################################################

cat("\n===== STEP3.2_03_clustering: Variables (OH/FP) =====\n")
for (ds in datasets_variables) {
  process_one_dataset_clust("variables", ds)
}

cat("\n===== STEP3.2_03_clustering: Samples (A/B/C/D) =====\n")
for (ds in datasets_samples) {
  process_one_dataset_clust("samples", ds)
}

cat("\n>>> STEP3.2_03_clustering finished (run_id = ", run_id, " )\n")


[INFO] Using override_run_id = 20251130_153500



>>> STEP3.2_03_clustering <U+2014> run_id: 20251130_153500 
>>> STEP3.2_03_clustering input dir:   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500 
>>> STEP3.2_03_clustering output dir:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/03_clustering_STEP3.2_signlessCorr/run_20251130_153500 

===== STEP3.2_03_clustering: Variables (OH/FP) =====



[STEP3.2_03] unit = variables, dataset = OH



  [INFO] Using coord_file (linear_top3): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/variables/OH/MDScoords_linear_top3_variables_OH_20251130_153500.csv

    [NbClust] index = silhouette

    [NbClust] index = dunn

Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
    [NbClust] index = gap

    [NbClust] index = ch

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points t

    [fixed-k] k = 10

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 2 rows containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 2 rows containing missing values or values outside the scale range (`geom_path()`)."
    [fixed-k] k = 25

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 8 rows containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 16 rows containing missing values or values outside the scale range (`geom_path()`)."
  [INFO] Using coord_file (nonlinear_cumeig): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/variables/OH/MDScoords_nonlinear_cumeig_variables_OH_20251130_153500.csv

    

...


[STEP3.2_03] unit = samples, dataset = A_OH_plus_others



  [INFO] Using coord_file (linear_top3): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/A_OH_plus_others/MDScoords_linear_top3_samples_A_OH_plus_others_20251130_153500.csv

    [NbClust] index = silhouette

    [NbClust] index = dunn

Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
    [NbClust] index = gap

    [NbClust] index = ch

Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 

    [NbClust] index = silhouette

    [NbClust] index = dunn

    [NbClust] index = gap

    [NbClust] index = ch

    [NbClust] index = db

    [NbClust] index = ptbiserial

    [fixed-k] k = 10

    [fixed-k] k = 25

    [fixed-k] k = 50

Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."

[STEP3.2_03] unit = samples, dataset = C_OH_only



  [INFO] Using coord_file (linear_top3): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/C_OH_only/MDScoords_linear_top3_samples_C_OH_only_20251130_153500.csv

    [NbClust] index = silhouette

    [NbClust] index = dunn

    [NbClust] index = gap

    [NbClust] index 

Too few points to calculate an ellipse
Warning message:
"Removed 5 rows containing missing values or values outside the scale range (`geom_path()`)."

[STEP3.2_03] unit = samples, dataset = D_FP_only



  [INFO] Using coord_file (linear_top3): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/D_FP_only/MDScoords_linear_top3_samples_D_FP_only_20251130_153500.csv

    [NbClust] index = silhouette

Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 2 is not positive"
Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 2 is not positive"
    [NbClust] index = dunn

Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 2 is not positive"
Warning

    [NbClust] index = dunn

Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 1 is not positive"
Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 1 is not positive"
    [NbClust] index = gap

    [NbClust] index = ch

Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 1 is not positive"
Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 1 is not positive"
    [NbClust] index = db

Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 1 is not positive"
Warning message:
"Computation failed in `stat_ellipse()`.
Caused by error in `chol.default()`:
! the leading minor of order 1 is not positive"
    [NbClust] index = ptbiserial

 


>>> STEP3.2_03_clustering finished (run_id =  20251130_153500  )


In [4]:
step02_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr"

list.dirs(step02_base, full.names = FALSE, recursive = FALSE)


[1] "run_20251128_193916" "run_20251128_194342" "run_20251128_195239"
[4] "run_20251128_195956"

In [5]:
step02_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr"

runs <- c("run_20251128_193916",
          "run_20251128_194342",
          "run_20251128_195239",
          "run_20251128_195956")

for (r in runs) {
  cat("\n=== ", r, " ===\n")
  print(list.dirs(file.path(step02_base, r),
                  full.names = FALSE, recursive = TRUE))
}



===  run_20251128_193916  ===
[1] ""             "variables"    "variables/FP" "variables/OH"

===  run_20251128_194342  ===
[1] ""          "samples"   "variables"

===  run_20251128_195239  ===
[1] ""          "samples"   "variables"

===  run_20251128_195956  ===
[1] ""          "samples"   "variables"


In [10]:
##図表pngのみ

# #!/usr/bin/env Rscript

# ############################################################
# # STEP3.2_03_clustering.R
# #  - Variables / Samples 両方
# #  - NbClust (k=2–25, 6 indices)
# #  - fixed-k clustering (k=10, 25, 50)
# #  - 入力: 02_mds_STEP3.2_signlessCorr の出力
# #  - 出力: 03_clustering_STEP3.2_signlessCorr
# ############################################################

# suppressPackageStartupMessages({
#   if (!require(NbClust))  { install.packages("NbClust");  library(NbClust) }
#   if (!require(ggplot2))  { install.packages("ggplot2");  library(ggplot2) }
#   if (!require(ggrepel))  { install.packages("ggrepel");  library(ggrepel) }
#   if (!require(scales))   { install.packages("scales");   library(scales)  }
#   if (!require(MASS))     { install.packages("MASS");     library(MASS)    }
#   if (!require(mclust))   { install.packages("mclust");   library(mclust)  }
#   if (!require(dplyr))    { install.packages("dplyr");    library(dplyr)   }
# })

# #### 0. パス設定 ####
# root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
# sub_base  <- file.path(root_base, "sub")

# step02_base <- file.path(sub_base, "02_mds_STEP3.2_signlessCorr")
# step03_base <- file.path(sub_base, "03_clustering_STEP3.2_signlessCorr")

# # 対象データセット
# datasets_variables <- c("OH", "FP")
# datasets_samples   <- c("A_OH_plus_others",
#                         "B_FP_plus_others",
#                         "C_OH_only",
#                         "D_FP_only")

# # NbClust インデックス
# index_list <- c("silhouette","dunn","gap","ch","db","ptbiserial")

# #### 1. run_id の取得 ####
# # ★ Jupyter から使うときはここを書き換える：MDS が存在する run_id を指定 ★
# # 例: run_20251130_153500 があるなら "20251130_153500"
# override_run_id <- "20251130_153500"  # ← ここは必要に応じて変更

# get_run_id <- function() {
#   # 1) 手動 override があればそれを最優先
#   if (!is.na(override_run_id) &&
#       is.character(override_run_id) &&
#       grepl("^\\d{8}_\\d{6}$", override_run_id)) {
#     message("[INFO] Using override_run_id = ", override_run_id)
#     return(override_run_id)
#   }

#   # 2) コマンドライン引数から取得 (Rscript 実行時)
#   args <- commandArgs(trailingOnly = TRUE)
#   if (length(args) >= 1) {
#     cand <- args[1]
#     if (grepl("^\\d{8}_\\d{6}$", cand)) {
#       message("[INFO] Using run_id from commandArgs = ", cand)
#       return(cand)
#     } else {
#       stop("Usage: Rscript STEP3.2_03_clustering.R <run_id (YYYYMMDD_HHMMSS)>\n",
#            "  - or set override_run_id <- \"YYYYMMDD_HHMMSS\" at the top of this script.")
#     }
#   }

#   stop("Usage: Rscript STEP3.2_03_clustering.R <run_id (YYYYMMDD_HHMMSS)>\n",
#        "  - or set override_run_id <- \"YYYYMMDD_HHMMSS\" at the top of this script.")
# }

# run_id <- get_run_id()

# cat(">>> STEP3.2_03_clustering — run_id:", run_id, "\n")
# input_run_dir  <- file.path(step02_base, paste0("run_", run_id))
# output_run_dir <- file.path(step03_base, paste0("run_", run_id))
# cat(">>> STEP3.2_03_clustering input dir:  ", input_run_dir,  "\n")
# cat(">>> STEP3.2_03_clustering output dir: ", output_run_dir, "\n")

# dir.create(output_run_dir, showWarnings = FALSE, recursive = TRUE)

# #### 2. Helper 関数 ####

# # NbClust 安全ラッパー
# run_nbclust_safe <- function(X, index_name) {
#   tryCatch({
#     NbClust(data = X,
#             diss = NULL,
#             distance = "euclidean",
#             min.nc = 2,
#             max.nc = min(25, nrow(X) - 1),
#             method = "ward.D2",
#             index = index_name)
#   }, error = function(e) {
#     warning("[NbClust failed] index = ", index_name, " : ", e$message)
#     return(NULL)
#   })
# }

# # MDS 散布図（NbClust & fixed-k 共通）
# plot_mds_scatter <- function(X, cluster_vec, title, subtitle, out_png) {
#   df_plot <- data.frame(
#     MDS1    = X[,1],
#     MDS2    = X[,2],
#     Cluster = factor(cluster_vec),
#     ID      = rownames(X),
#     stringsAsFactors = FALSE
#   )

#   rng <- range(c(df_plot$MDS1, df_plot$MDS2), na.rm = TRUE)
#   pad <- diff(rng) * 0.05
#   lims <- c(min(rng) - pad, max(rng) + pad)

#   p_sc <- ggplot(df_plot, aes(MDS1, MDS2, color = Cluster)) +
#     stat_ellipse(aes(group = Cluster),
#                  type = "norm", level = 0.95,
#                  linewidth = 0.6, alpha = 0.25) +
#     geom_point(size = 1.6, alpha = 0.9) +
#     coord_equal(xlim = lims, ylim = lims, expand = TRUE) +
#     scale_color_hue(h = c(0, 360), l = 55, c = 90) +
#     labs(title = title,
#          subtitle = subtitle,
#          x = "Dim 1", y = "Dim 2", color = "Cluster") +
#     theme_minimal(base_size = 11) +
#     theme(
#       panel.grid.major = element_line(linewidth = 0.3, linetype = "dashed"),
#       panel.grid.minor = element_blank(),
#       axis.title       = element_text(face = "bold"),
#       plot.title       = element_text(face  = "bold")
#     )

#   ggsave(out_png, p_sc, width = 7, height = 6, dpi = 300)
# }

# # 1 Dataset (unit × ds) 分のクラスタリング
# process_one_dataset_clust <- function(unit, ds) {

#   message("\n[STEP3.2_03] unit = ", unit, ", dataset = ", ds, "\n")

#   # 02_mds 側のディレクトリ
#   mds_dir_ds <- file.path(input_run_dir, unit, ds)
#   if (!dir.exists(mds_dir_ds)) {
#     stop("[ERROR] MDS dir not found: ", mds_dir_ds)
#   }

#   # 03_clustering 側の出力ディレクトリ
#   out_ds_base <- file.path(output_run_dir, unit, ds)
#   dir.create(out_ds_base, showWarnings = FALSE, recursive = TRUE)

#   labels_dir  <- file.path(out_ds_base, "labels")
#   indices_dir <- file.path(out_ds_base, "indices")
#   plots_dir   <- file.path(out_ds_base, "plots")
#   dir.create(labels_dir,  showWarnings = FALSE, recursive = TRUE)
#   dir.create(indices_dir, showWarnings = FALSE, recursive = TRUE)
#   dir.create(plots_dir,   showWarnings = FALSE, recursive = TRUE)

#   # mode_dim 条件
#   cond_list <- c("linear_top3","linear_cumeig","nonlinear_top3","nonlinear_cumeig")

#   label_df <- NULL
#   ksum     <- data.frame(
#     mode     = character(0),
#     dim_mode = character(0),
#     index    = character(0),
#     k        = integer(0),
#     stringsAsFactors = FALSE
#   )

#   #---------------------------
#   # 各 mode_dim ごとの処理
#   #---------------------------
#   for (cond in cond_list) {

#     parts <- strsplit(cond, "_")[[1]]
#     mode     <- parts[1]   # "linear" or "nonlinear"
#     dim_mode <- parts[2]   # "top3" or "cumeig"

#     #------------------------------------
#     #  座標ファイル名の候補 (2パターン)
#     #------------------------------------
#     # 候補1: MDScoords_linear_top3_samples_A_OH_plus_others_2025...csv
#     cand1 <- file.path(
#       mds_dir_ds,
#       sprintf("MDScoords_%s_%s_%s_%s_%s.csv",
#               mode, dim_mode, unit, ds, run_id)
#     )

#     # 候補2: MDS_linear_top3_A_OH_plus_others_2025...csv
#     cond_tag <- paste(mode, dim_mode, sep = "_")
#     cand2 <- file.path(
#       mds_dir_ds,
#       sprintf("MDS_%s_%s_%s.csv",
#               cond_tag, ds, run_id)
#     )

#     coord_file <- NA_character_
#     for (f in c(cand1, cand2)) {
#       if (file.exists(f)) {
#         coord_file <- f
#         break
#       }
#     }

#     # ★★★ ここを変更：見つからない場合はスキップ ★★★
#     if (is.na(coord_file)) {
#       warning(
#         sprintf(
#           "[WARN] coord file not found for unit=%s, ds=%s, cond=%s; skip this condition.\n  tried:\n    %s\n    %s",
#           unit, ds, cond, cand1, cand2
#         )
#       )
#       next  # この mode_dim はスキップして次へ
#     }

#     message("  [INFO] Using coord_file (", cond, "): ", coord_file)
#     X <- as.matrix(read.csv(coord_file, row.names = 1, check.names = FALSE))

#     # 初回のみ ID ベクトルを確定
#     if (is.null(label_df)) {
#       id_vec  <- rownames(X)
#       label_df <- data.frame(id = id_vec, stringsAsFactors = FALSE)
#     }

#     ##==========================
#     ## NbClust によるクラスタリング
#     ##==========================
#     for (idx in index_list) {

#       message("    [NbClust] index = ", idx)
#       res <- run_nbclust_safe(X, idx)

#       col_name <- paste(cond, idx, sep = "_")

#       if (is.null(res)) {
#         # NbClust 失敗 → NA 埋め
#         label_df[[col_name]] <- NA_integer_
#         ksum <- rbind(
#           ksum,
#           data.frame(
#             mode     = mode,
#             dim_mode = dim_mode,
#             index    = idx,
#             k        = NA_integer_,
#             stringsAsFactors = FALSE
#           )
#         )
#         next
#       }

#       # --- ベストパーティション & k ---
#       grp <- as.factor(res$Best.partition)
#       # id の順序で並べ直し
#       grp <- grp[label_df$id]
#       label_df[[col_name]] <- as.integer(grp)

#       # k の決定
#       k_val <- NA_integer_
#       if (!is.null(res$Best.nc)) {
#         if (is.list(res$Best.nc) && !is.null(res$Best.nc[["Number_clusters"]])) {
#           k_val <- as.integer(res$Best.nc[["Number_clusters"]])
#         } else if (is.matrix(res$Best.nc) && "Number_clusters" %in% rownames(res$Best.nc)) {
#           k_val <- as.integer(res$Best.nc["Number_clusters", 1])
#         } else if (is.numeric(res$Best.nc)) {
#           k_val <- as.integer(res$Best.nc[1])
#         }
#       }

#       ksum <- rbind(
#         ksum,
#         data.frame(
#           mode     = mode,
#           dim_mode = dim_mode,
#           index    = idx,
#           k        = k_val,
#           stringsAsFactors = FALSE
#         )
#       )

#       # --- ラベル CSV 出力 (NbClust) ---
#       out_lab_file <- file.path(
#         labels_dir,
#         sprintf("ClusterAssign_%s_%s_%s_%s_%s_%s.csv",
#                 mode, dim_mode, idx, unit, ds, run_id)
#       )
#       write.csv(
#         data.frame(ID = label_df$id, Cluster = grp),
#         out_lab_file,
#         row.names = FALSE
#       )

#       # --- k vs score の CSV & プロット (NbClust::All.index) ---
#       if (!is.null(res$All.index)) {
#         idx_vals  <- as.numeric(res$All.index)
#         k_labels  <- names(res$All.index)
#         k_seq <- if (is.null(k_labels) || any(k_labels == "")) {
#           seq_along(idx_vals)
#         } else {
#           suppressWarnings(as.numeric(k_labels))
#         }
#         if (any(is.na(k_seq))) k_seq <- seq_along(idx_vals)

#         df_k_score <- data.frame(k = k_seq, score = idx_vals)
#         out_k_csv  <- file.path(
#           indices_dir,
#           sprintf("k_score_%s_%s_%s_%s_%s_%s.csv",
#                   mode, dim_mode, idx, unit, ds, run_id)
#         )
#         write.csv(df_k_score, out_k_csv, row.names = FALSE)

#         df_idx <- data.frame(k = k_seq, value = idx_vals)
#         p_idx <- ggplot(df_idx, aes(k, value)) +
#           geom_line(linewidth = 1) +
#           geom_point(size = 2) +
#           labs(
#             title    = paste0("NbClust All.index — ", idx),
#             subtitle = paste0(mode, " / ", dim_mode,
#                               " | unit=", unit, ", ds=", ds),
#             x = "k", y = "Index value"
#           ) +
#           theme_minimal(base_size = 11) +
#           theme(
#             panel.grid.minor = element_blank(),
#             axis.title       = element_text(face = "bold"),
#             plot.title       = element_text(face = "bold")
#           )

#         out_idx_png <- file.path(
#           plots_dir,
#           sprintf("NbClust_AllIndex_%s_%s_%s_%s_%s_%s.png",
#                   mode, dim_mode, idx, unit, ds, run_id)
#         )
#         ggsave(out_idx_png, p_idx, width = 7, height = 5, dpi = 300)
#       }

#       # --- MDS 散布図 (NbClust) ---
#       out_mds_png <- file.path(
#         plots_dir,
#         sprintf("MDS12_scatter_%s_%s_%s_%s_%s_%s.png",
#                 mode, dim_mode, idx, unit, ds, run_id)
#       )

#       plot_mds_scatter(
#         X,
#         cluster_vec = grp,
#         title    = sprintf("MDS (%s / %s) — Ward.D2 × NbClust", mode, dim_mode),
#         subtitle = sprintf("Index: %s | k = %s | unit=%s, ds=%s",
#                            idx, ifelse(is.na(k_val), "NA", k_val), unit, ds),
#         out_png  = out_mds_png
#       )
#     } # end for idx in index_list

#     ##==========================
#     ## fixed-k クラスタリング
#     ##==========================
#     message("    [fixed-k] k = 10, 25, 50")

#     hc <- hclust(dist(X), method = "ward.D2")

#     for (k_fixed in c(10, 25, 50)) {
#       if (nrow(X) <= k_fixed) {
#         warning("[fixed-k] nrow(X) = ", nrow(X),
#                 " <= k_fixed = ", k_fixed,
#                 " → skip.")
#         next
#       }

#       cl_fix <- cutree(hc, k = k_fixed)

#       # ラベル CSV (fixed-k)
#       out_lab_k <- file.path(
#         labels_dir,
#         sprintf("ClusterAssign_%s_%s_k%d_%s_%s_%s.csv",
#                 mode, dim_mode, k_fixed, unit, ds, run_id)
#       )
#       write.csv(
#         data.frame(ID = rownames(X), Cluster = cl_fix),
#         out_lab_k,
#         row.names = FALSE
#       )

#       # MDS 散布図 (fixed-k)
#       out_mds_k <- file.path(
#         plots_dir,
#         sprintf("MDS12_scatter_%s_%s_k%d_%s_%s_%s.png",
#                 mode, dim_mode, k_fixed, unit, ds, run_id)
#       )

#       plot_mds_scatter(
#         X,
#         cluster_vec = cl_fix,
#         title    = sprintf("MDS (%s / %s) — Ward.D2 (fixed-k)", mode, dim_mode),
#         subtitle = sprintf("k = %d | unit=%s, ds=%s",
#                            k_fixed, unit, ds),
#         out_png  = out_mds_k
#       )
#     } # end for k_fixed
#   }   # end for cond

#   ##==========================
#   ## label matrix & k-summary の保存
#   ##==========================
#   if (!is.null(label_df)) {
#     out_label_mat <- file.path(
#       out_ds_base,
#       sprintf("cluster_labels_matrix_%s_%s_%s.csv", unit, ds, run_id)
#     )
#     out_ksum <- file.path(
#       out_ds_base,
#       sprintf("cluster_k_summary_%s_%s_%s.csv", unit, ds, run_id)
#     )

#     write.csv(label_df, out_label_mat, row.names = FALSE)
#     write.csv(ksum,     out_ksum,     row.names = FALSE)

#     ##==========================
#     ## ARI: top3 vs cumeig (linear / nonlinear)
#     ##==========================
#     for (mode in c("linear","nonlinear")) {
#       ari_rows <- lapply(index_list, function(ix) {
#         a <- label_df[[paste0(mode, "_top3_", ix)]]
#         b <- label_df[[paste0(mode, "_cumeig_", ix)]]
#         if (is.null(a) || is.null(b) || all(is.na(a)) || all(is.na(b))) {
#           data.frame(Index = ix, ARI_top3_vs_cumeig = NA_real_)
#         } else {
#           data.frame(
#             Index               = ix,
#             ARI_top3_vs_cumeig  = mclust::adjustedRandIndex(as.factor(a),
#                                                             as.factor(b))
#           )
#         }
#       })
#       df_ari <- dplyr::bind_rows(ari_rows)

#       out_ari <- file.path(
#         out_ds_base,
#         sprintf("ARI_top3_vs_cumeig_%s_%s_%s_%s.csv", mode, unit, ds, run_id)
#       )
#       write.csv(df_ari, out_ari, row.names = FALSE)
#     }
#   } else {
#     warning("[WARN] No coordinate conditions were available for unit=", unit,
#             ", ds=", ds, " → no label matrix / ARI written.")
#   }

#   invisible(NULL)
# }

# #### 3. メイン処理 ####

# cat("\n===== STEP3.2_03_clustering: Variables (OH/FP) =====\n")
# for (ds in datasets_variables) {
#   process_one_dataset_clust("variables", ds)
# }

# cat("\n===== STEP3.2_03_clustering: Samples (A/B/C/D) =====\n")
# for (ds in datasets_samples) {
#   process_one_dataset_clust("samples", ds)
# }

# cat("\n>>> STEP3.2_03_clustering 完了 (run_id = ", run_id, ")\n\n")


[INFO] Using override_run_id = 20251130_153500



>>> STEP3.2_03_clustering <U+2014> run_id: 20251130_153500 
>>> STEP3.2_03_clustering input dir:   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500 
>>> STEP3.2_03_clustering output dir:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/03_clustering_STEP3.2_signlessCorr/run_20251130_153500 

===== STEP3.2_03_clustering: Variables (OH/FP) =====



[STEP3.2_03] unit = variables, dataset = OH


  [INFO] Using coord_file (linear_top3): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/variables/OH/MDScoords_linear_top3_variables_OH_20251130_153500.csv

    [NbClust] index = silhouette

    [NbClust] index = dunn

Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
    [NbClust] index = gap

    [NbClust] index = ch

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning me

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 10 rows containing missing values or values outside the scale range (`geom_path()`)."
    [NbClust] index = db

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 10 rows containing missing values or values outside the scale range (`geom_path()`)."
    [NbClust] index = ptbiserial

Too few points to calculate an ellipse
Warning messag

Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 28 rows containing missing values or values outside the scale range (`geom_path()`)."
  [INFO] Using coord_file (linear_cumeig): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3

    [fixed-k] k = 10, 25, 50

Too few points to calculate an ellipse
Warning message:
"Removed 1 row containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Warning message:
"Removed 8 rows containing missing values or values outside the scale range (`geom_path()`)."
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few points to calculate an ellipse
Too few 


===== STEP3.2_03_clustering: Samples (A/B/C/D) =====



[STEP3.2_03] unit = samples, dataset = A_OH_plus_others


Warning message in process_one_dataset_clust("samples", ds):
"[WARN] coord file not found for unit=samples, ds=A_OH_plus_others, cond=linear_top3; skip this condition.
  tried:
    /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/A_OH_plus_others/MDScoords_linear_top3_samples_A_OH_plus_others_20251130_153500.csv
    /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/A_OH_plus_others/MDS_linear_top3_A_OH_plus_others_20251130_153500.csv"
Warning message in process_one_dataset_clust("samples", ds):
"[WARN] coord file not found for unit=samples, ds=A_OH_plus_others, cond=linear_cumeig; skip this condition.
  tried:
    /Volumes/csbdeep11/_yasu-i/20250303_rebuiled

    [fixed-k] k = 10, 25, 50


[STEP3.2_03] unit = samples, dataset = D_FP_only


Warning message in process_one_dataset_clust("samples", ds):
"[WARN] coord file not found for unit=samples, ds=D_FP_only, cond=linear_top3; skip this condition.
  tried:
    /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/D_FP_only/MDScoords_linear_top3_samples_D_FP_only_20251130_153500.csv
    /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/samples/D_FP_only/MDS_linear_top3_D_FP_only_20251130_153500.csv"
Warning message in process_one_dataset_clust("samples", ds):
"[WARN] coord file not found for unit=samples, ds=D_FP_only, cond=linear_cumeig; skip this condition.
  tried:
    /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/202


>>> STEP3.2_03_clustering <U+5B8C><U+4E86> (run_id =  20251130_153500 )

